# cmri_model — Figure Path Check & Visual Gallery (Stage 1 Only)

**Only two tasks:** (1) check all known HPC locations; (2) **display each recovered PNG/JPG/SVG inline** in this notebook so you can photograph and decide clusters. No model inference, no training, no renaming, no recomputation, no SVG export, and no changes to source files.

Use the **Python (heart_gpu)** kernel and choose **Run → Run All Cells**. Scroll through the rendered images; each image has an ID and exact original path. Repeated identical images from multiple locations are shown once.

In [ ]:
from pathlib import Path
from IPython.display import display, Image, SVG, Markdown
from PIL import Image as PILImage
from collections import defaultdict
import hashlib

PROJECT = Path('/home/ypan/heart_liver/model_train/heart-molecular-pretraining')
EVAL = Path('/project/yli4/ypan/heart_evaluation/general_audit_10k_v2')
PRUN = Path('/project/yli4/ypan/runs/evaluation')
HRUN = PROJECT / 'runs/evaluation'

# Each root is checked individually. Directory names are unchanged.
SOURCES = [
 ('Recovered originals', PROJECT/'summary_figures/recovered_original_figures', True),
 ('Archived general-pretraining v3', PROJECT/'summary_figures/archived_general_pretraining', True),
 ('H200 early n192 — project', PRUN/'h200_best_embeddings/full_n192_seed17', True),
 ('H200 early n192 — home', HRUN/'h200_best_embeddings/full_n192_seed17', True),
 ('T4 early n192 — project', PRUN/'t4_best_embeddings/full_n192_seed17', True),
 ('T4 early n192 — home', HRUN/'t4_best_embeddings/full_n192_seed17', True),
 ('Old embeddings — project', PRUN/'old_best_embeddings', True),
 ('H200 validation 10K', EVAL/'checkpoint_evaluation/full_10k_batch8', True),
 ('H200 training history', EVAL/'training_history', True),
 ('Chemistry evidence', EVAL/'chemistry_slide_evidence', True),
 ('Duplicate audit', Path('/project/yli4/ypan/audits'), True),
 ('New summary composites', PROJECT/'summary_figures', False),
]

EXPECTED = [
 '01_atom_and_geometry_validation.png',
 '01b_detailed_validation_diagnostics.png',
 '02_atom_accuracy_by_element.png',
 '03_pca_chemistry_colored_four_panels.png',
 '04_benchmark_esol.png',
 '04_benchmark_freesolv.png',
 '04_benchmark_lipophilicity.png',
 '04_esol_labeled_data_efficiency.png',
 '04_lipophilicity_labeled_data_efficiency.png',
 '05_dataset_duplicate_counts.png',
 'CLS_raw_PCA_atomNum.png',
 'CLS_raw_PCA.png',
]
IMAGE_EXTS = {'.png','.jpg','.jpeg','.svg','.webp'}

def get_images(folder, recursive):
    if not folder.is_dir(): return []
    try:
        entries = folder.rglob('*') if recursive else folder.iterdir()
        return sorted((p for p in entries if p.is_file() and p.suffix.lower() in IMAGE_EXTS
                       and not any(part.startswith('smoke_') for part in p.parts)), key=lambda p:str(p))
    except (OSError, PermissionError) as exc:
        print('Cannot search:',folder,exc)
        return []

print('Project root:', PROJECT)
print('Project accessible:', PROJECT.is_dir())
print('This notebook does not execute any evaluation code.')

## 1. Check every source directory and expected original figure

In [ ]:
FOUND = []
for label,folder,recursive in SOURCES:
    images = get_images(folder,recursive)
    FOUND.append((label,folder,images))
    print(f"{'FOUND' if folder.is_dir() else 'NOT FOUND':9} | {len(images):3} image(s) | {label} | {folder}")

original_dir = PROJECT/'summary_figures/recovered_original_figures'
archive_dir = PROJECT/'summary_figures/archived_general_pretraining'
print('\n=== Expected 12 restored PNG files ===')
for name in EXPECTED:
    candidates = [original_dir/name]
    if archive_dir.is_dir(): candidates.extend(archive_dir.rglob(name))
    match=next((p for p in candidates if p.is_file()),None)
    print(('FOUND    ' if match else 'NOT FOUND'),name, '|', str(match) if match else '')

print('\n=== Saved numerical sources (presence only) ===')
for path in [
 EVAL/'checkpoint_evaluation/full_10k_batch8/evaluation_summary.json',
 EVAL/'checkpoint_evaluation/full_10k_batch8/molecule_diagnostics.csv',
 EVAL/'checkpoint_evaluation/full_10k_batch8/atom_token_class_accuracy.csv',
 EVAL/'chemistry_slide_evidence/independent_property_benchmarks/benchmark_all_runs.csv',
 PRUN/'h200_best_embeddings/full_n192_seed17/evaluation_summary.json',
 HRUN/'h200_best_embeddings/full_n192_seed17/evaluation_summary.json',
]:
    print(('FOUND    ' if path.is_file() else 'NOT FOUND'),path)

## 2. Actual visual gallery — every discovered image

The images below are **rendered in Jupyter output**, not just their filenames. They are grouped by location. Images with identical bytes are shown only once; duplicate paths are printed below. Use the image **ID** in your photos or future instructions. To prevent redundant gallery pages, archived originals are shown once, even if duplicated in recovered originals.

In [ ]:
seen = {}                  # digest -> first ID
picture_count = 0
index = []
for label,folder,files in FOUND:
    display(Markdown(f'## {label}'))
    print('Path:',folder)
    if not files:
        print('(No images at this location, or directory not found.)')
        continue
    this_count=0
    for p in files:
        try:
            digest=hashlib.sha256(p.read_bytes()).hexdigest()
            if digest in seen:
                print('DUPLICATE:',p.name,'— same content as',seen[digest], '|',p)
                continue
            picture_count+=1
            this_count+=1
            picid=f'FIG-{picture_count:03d}'
            seen[digest]=picid
            index.append((picid,label,p.name,str(p)))
            display(Markdown(f'### {picid} | {p.name}'))
            print('Original path:',p)
            if p.suffix.lower()=='.svg':
                display(SVG(filename=str(p)))
            else:
                # Inline display, scaled for readable screenshots; does not alter original image.
                display(Image(filename=str(p), width=1050))
        except Exception as exc:
            print('UNABLE TO DISPLAY:',p,'|',type(exc).__name__,str(exc))
    print(f'{this_count} unique image(s) rendered from {label}')
print(f'\nTOTAL UNIQUE IMAGES ACTUALLY RENDERED: {picture_count}')

## 3. Figure index for your cluster selection

In [ ]:
print(f'{"ID":9} | {"GROUP":28} | FILE')
print('-'*112)
for picid,label,name,path in index:
    print(f'{picid:9} | {label[:28]:28} | {name}')
print('\nPlease photograph the figures you want together and tell me their IDs.')